# Simulating Dense Polymer Melts

This notebook starts from the `.mupt.sdf` written by `Building_Dense_Polymer_Melts.ipynb`, loads it back into MuPT, parameterizes it with OpenFF, runs a minimal OpenMM workflow, and analyzes the trajectory with MDAnalysis.

The goal is not to teach OpenMM internals. The goal is to show that MuPT-built dense melts can be handed off to familiar MD tools with a few user-facing MD parameters.

In [ ]:
from pathlib import Path
import sys

EXAMPLES_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'examples_system').is_dir() and (candidate / 'recipes').is_dir()
)
if str(EXAMPLES_ROOT) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_ROOT))

from mupt.temporary.sdf import primitive_from_mupt_sdf

from utilities.notebook import (
    load_manifest,
)
from utilities.simulation import (
    MDRunConfig,
    build_openff_interchange,
    run_openmm_workflow,
)
from utilities.visualization import (
    plot_openmm_state_data,
    plot_radius_of_gyration,
)

OUTPUT_DIR = EXAMPLES_ROOT / 'examples_system' / 'dense_melt_demo_outputs'
MANIFEST_PATH = OUTPUT_DIR / 'dense_melt_manifest.json'

## Load the MuPT SDF

The temporary `.mupt.sdf` file preserves one polymer chain per SDF record plus MuPT atom metadata.
`mupt_universe` is the MuPT representation we built in the previous notebook, read into memory from file.

In [ ]:
manifest = load_manifest(MANIFEST_PATH)
sdf_path = Path(manifest['sdf_path'])
resname_map = manifest['resname_map']

mupt_universe = primitive_from_mupt_sdf(sdf_path)
print(mupt_universe.hierarchy_summary(to_depth=2))
print(f'Loaded {sdf_path}')

## Build an OpenFF Interchange

The default force field and charge model are the specific workflow we want to showcase: OpenFF Sage plus NAGL/AshGC charges. The `MDRunConfig` below also sets the short OpenMM schedule used in the next cell: minimization, then NVT for `nvt_time_ns`, then immediate NPT for `npt_time_ns`. The helper writes separate NVT and NPT DCD and state-data files.

You can write your own OpenMM functions if you so choose, this is just a demo.

TODO: Decide if we want to support direct MuPT -> OpenFF Interchange.

In [ ]:
md_config = MDRunConfig(
    temperature_k=450.0,
    pressure_atm=1.0,
    timestep_fs=2.0,
    nvt_time_ns=0.10,
    npt_time_ns=0.30,
    frames_to_save=50,
    force_field='openff-2.2.1.offxml',
    charge_method='openff-gnn-am1bcc-1.0.0.pt',
)

interchange = build_openff_interchange(
    mupt_universe,
    resname_map=resname_map,
    force_field=md_config.force_field,
    charge_method=md_config.charge_method,
)
print(f'OpenFF Interchange ready with {interchange.topology.n_atoms} atoms.')

## Export Other MD Engine Starting Files

Once MuPT has handed the system to OpenFF Interchange, writing starting files for other MD engines is straightforward. The files below are useful starting points for GROMACS and LAMMPS workflows; inspect and adapt the generated run/input settings before production simulations.

The GROMACS export merges equivalent atom types by default. This produces a smaller topology and substantially reduces GROMACS preprocessing time. Interchange currently marks `_merge_atom_types` as a provisional option, so this call may need an update when the pinned Interchange version changes.

In [ ]:
engine_dir = OUTPUT_DIR / 'engine_starting_files'
gromacs_dir = engine_dir / 'gromacs'
lammps_dir = engine_dir / 'lammps'
gromacs_dir.mkdir(parents=True, exist_ok=True)
lammps_dir.mkdir(parents=True, exist_ok=True)

gromacs_prefix = gromacs_dir / manifest['recipe_name']
lammps_prefix = lammps_dir / manifest['recipe_name']
MERGE_GROMACS_ATOM_TYPES = True
interchange.to_gromacs(
    str(gromacs_prefix),
    _merge_atom_types=MERGE_GROMACS_ATOM_TYPES,
)
interchange.to_lammps(str(lammps_prefix))

engine_files = {
    'GROMACS coordinates': gromacs_prefix.with_suffix('.gro'),
    'GROMACS topology': gromacs_prefix.with_suffix('.top'),
    'GROMACS point-energy MDP': gromacs_dir / f"{manifest['recipe_name']}_pointenergy.mdp",
    'LAMMPS data': lammps_prefix.with_suffix('.lmp'),
    'LAMMPS point-energy input': lammps_dir / f"{manifest['recipe_name']}_pointenergy.in",
}
for label, path in engine_files.items():
    print(f'{label}: {path}')

## Run Minimal OpenMM MD

This helper runs energy minimization, short NVT, then short NPT. Increase the nanosecond values after confirming the default works on your machine.

In [ ]:
openmm_result = run_openmm_workflow(
    interchange,
    config=md_config,
    output_dir=OUTPUT_DIR,
    prefix=manifest['recipe_name'],
)
print(f'Initial energy:   {openmm_result.initial_energy}')
print(f'Minimized energy: {openmm_result.minimized_energy}')
print(f'Final energy:     {openmm_result.final_energy}')
print(f'NVT trajectory:   {openmm_result.nvt_trajectory_path}')
print(f'NPT trajectory:   {openmm_result.npt_trajectory_path}')
print(f'NVT state data:   {openmm_result.nvt_state_data_path}')
print(f'NPT state data:   {openmm_result.npt_state_data_path}')
if 'visualization_pdb' in manifest:
    print(f"Visualization PDB: {manifest['visualization_pdb']}")
    print('Load this PDB and either DCD into PyMOL/VMD/Ovito to watch the movie.')

## Watch the NPT Trajectory

The OpenMM helper wrote a DCD trajectory for the NPT segment. MDTraj loads that trajectory with the OpenFF/OpenMM topology, and NGLView displays it as an interactive movie in the notebook.

In [ ]:
# NGLView requires a notebook frontend with Jupyter widget and JavaScript support.
try:
    import mdtraj
    import nglview as nv
    from IPython.display import display

    trajectory = mdtraj.load(
        str(openmm_result.npt_trajectory_path),
        top=mdtraj.Topology.from_openmm(interchange.to_openmm_topology()),
    )
    view = nv.show_mdtraj(trajectory)
    view.clear_representations()
    view.add_representation('licorice')
    view.add_unitcell()
    view.center()
    print(f'If the widget does not render, open this trajectory in an external viewer: {openmm_result.npt_trajectory_path}')
    display(view)
except Exception as exc:
    print(f'Interactive view unavailable: {exc}')
    print(f'Open this trajectory in an external viewer: {openmm_result.npt_trajectory_path}')

## Inspect OpenMM State Data

Before loading the trajectory into MDAnalysis, check the OpenMM reporter output. The plots below show density and potential energy from the NPT segment only.

In [ ]:
state_df, fig, axes = plot_openmm_state_data(openmm_result.npt_state_data_path)
state_df.head()

## Analyze with MDAnalysis

MuPT can also create an MDAnalysis Universe directly from the role-aware hierarchy. We load in the OpenMM trajectory we just wrote and can compute radius of gyration as a function of time.

The default system with very short MD here is a demonstration of the workflow, not a converged production simulation. You can run a larger system, or a longer simulation if you'd like. Any analyses you run on a MDAnalysis Universe can be run with the outputs from MuPT.

In [ ]:
from rdkit import Chem
import pandas as pd

from mupt.interfaces.mdanalysis import primitive_to_mdanalysis

mda_universe = primitive_to_mdanalysis(mupt_universe, resname_map=resname_map)
mda_universe.load_new(str(openmm_result.npt_trajectory_path))

periodic_table = Chem.GetPeriodicTable()
masses = [periodic_table.GetAtomicWeight(element) for element in mda_universe.atoms.elements]
mda_universe.add_TopologyAttr('masses', masses)

atoms = mda_universe.select_atoms('all')
rg_df = pd.DataFrame(
    {
        'frame': int(ts.frame),
        'time_ps': float(getattr(ts, 'time', ts.frame)),
        'rg_angstrom': float(atoms.radius_of_gyration()),
        'rg_nm': float(atoms.radius_of_gyration()) / 10.0,
    }
    for ts in mda_universe.trajectory
)
rg_df.head()

In [ ]:
plot_radius_of_gyration(rg_df)

## Next Experiments

- Increase `nvt_time_ns` and `npt_time_ns`.
- Switch to a larger melt from the build notebook.
- Compute Rg per chain instead of all atoms.
- Compare long, equilibrated polystyrene simulations to literature Rg scaling.